In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
import torchvision.models as models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

data = "/content/drive/MyDrive/dataset/dogscats/valid"

Mounted at /content/drive


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [ ]:

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 140MB/s]


In [ ]:

loss_fxn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)


In [ ]:

transformation = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std  = [0.229, 0.224, 0.225]
    )
])

dataset = datasets.ImageFolder(root=data, transform=transformation)



In [ ]:
train_set = int(0.8 * len(dataset))
test_set = len(dataset) - train_set
print(f"Train size: {train_set}, Test size: {test_set}")


Train size: 1608, Test size: 402


In [ ]:
train_dataset, test_dataset = random_split(dataset, [train_set, test_set])
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [ ]:

loss_his = []
acc_his = []

for epoch in range(1, 11):
    model.train()
    batch_loss = 0
    correct = 0
    total = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        output = model(image)
        loss = loss_fxn(output, label)
        batch_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        predict = torch.argmax(output, 1)
        correct += (predict == label).sum().item()
        total += label.size(0)

    epoch_loss = batch_loss / len(train_loader)
    epoch_acc = (correct / total) * 100
    loss_his.append(epoch_loss)
    acc_his.append(epoch_acc)
    print(f"Epoch {epoch}  Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%")

    model.eval()
    test_batch_loss = 0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for image, label in test_loader:
            image = image.to(device)
            label = label.to(device)

            output = model(image)

            loss = loss_fxn(output, label)
            predict = torch.argmax(output, dim=1)

            test_batch_loss += loss.item()
            test_total += label.size(0)
            test_correct += (predict == label).sum().item()

    test_loss = test_batch_loss / len(test_loader)
    test_acc = 100 * (test_correct / test_total)
    print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%\n")


Epoch 1  Train Loss: 0.2604, Train Acc: 90.30%
Test Loss: 0.1228, Test Acc: 95.52%

Epoch 2  Train Loss: 0.1734, Train Acc: 93.16%
Test Loss: 0.0771, Test Acc: 98.51%

Epoch 3  Train Loss: 0.1419, Train Acc: 94.22%
Test Loss: 0.0647, Test Acc: 98.51%

Epoch 4  Train Loss: 0.1396, Train Acc: 94.09%
Test Loss: 0.0704, Test Acc: 97.51%

Epoch 5  Train Loss: 0.1037, Train Acc: 96.08%
Test Loss: 0.0611, Test Acc: 98.01%

Epoch 6  Train Loss: 0.1079, Train Acc: 95.52%
Test Loss: 0.0528, Test Acc: 98.26%

Epoch 7  Train Loss: 0.1160, Train Acc: 95.02%
Test Loss: 0.0472, Test Acc: 98.51%

Epoch 8  Train Loss: 0.0937, Train Acc: 96.27%
Test Loss: 0.0493, Test Acc: 97.76%

Epoch 9  Train Loss: 0.1059, Train Acc: 95.77%
Test Loss: 0.0541, Test Acc: 98.01%

Epoch 10  Train Loss: 0.0883, Train Acc: 96.95%
Test Loss: 0.0532, Test Acc: 97.76%

